# Creating the model with CO2 limit

in this notebook, the model created in model.ipynb will be imported and global_constraints of CO2 limit will be applied.

## Importing packages

In [3]:
import linopy
import pypsa

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import cartopy
import cartopy.crs as ccrs

import networkx as nx

import atlite
from atlite.gis import ExclusionContainer, shape_availability
from rasterio.plot import show
from rasterio.crs import CRS
import rasterio as rio

from pathlib import Path
import xarray as xr

## Importing the base model

In [4]:
n_no_co2 = pypsa.Network("pypsa_model_n.nc") # the model without CO2 limits is imported

INFO:pypsa.network.io:New version 1.0.7 available! (Current: 1.0.5)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, sub_networks


In [5]:
n_no_co2.global_constraints

attribute,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,


## Adding co2 limit constraint

In [6]:
n_no_co2.add(
    "GlobalConstraint",
    "CO2Limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=0, # the CO2 limit is 0 --> 100% renewable system
)

In [7]:
n_no_co2.global_constraints

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
CO2Limit,primary_energy,NaN,,co2_emissions,<=,0.0,0.0


## Solving the model

In [11]:
# solve the model with co2 limit
n_no_co2.optimize(
    #snapshots=n_no_co2.snapshots[:168],  # 1 week
    solver_name="gurobi",
    log_to_console=False
)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 8/8 [00:00<00:00, 50.78it/s]
INFO:linopy.io: Writing time: 3.91s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2750042


INFO:gurobipy:Set parameter LicenseID to value 2750042


...

Set parameter LogToConsole to value 0


INFO:gurobipy:Set parameter LogToConsole to value 0
INFO:gurobipy:Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 25.2.0 25C56)
INFO:gurobipy:
INFO:gurobipy:CPU model: Apple M4 Pro
INFO:gurobipy:Thread count: 14 physical cores, 14 logical processors, using up to 14 threads
INFO:gurobipy:
INFO:gurobipy:Non-default parameters:
INFO:gurobipy:LogToConsole  0
INFO:gurobipy:
INFO:gurobipy:Optimize a model with 5019879 rows, 2193249 columns and 10762913 nonzeros (Min)
INFO:gurobipy:Model fingerprint: 0x32dd7df5
INFO:gurobipy:Model has 318609 linear objective coefficients
INFO:gurobipy:Coefficient statistics:
INFO:gurobipy:  Matrix range     [2e-10, 7e+02]
INFO:gurobipy:  Objective range  [1e-02, 2e+05]
INFO:gurobipy:  Bounds range     [0e+00, 0e+00]
INFO:gurobipy:  RHS range        [2e+00, 4e+05]
INFO:gurobipy:Warning: Model contains large matrix coefficient range
INFO:gurobipy:         Consider reformulating model or setting NumericFocus parameter
INFO:gurobipy:        


Interrupt request received


INFO:gurobipy:
INFO:gurobipy:
INFO:gurobipy:Stopped in 0 iterations and 702.38 seconds (1277.40 work units)
INFO:gurobipy:Solve interrupted
Status: aborted
Termination condition: user_interrupt
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: 11



('aborted', 'user_interrupt')

In [ ]:
n_no_co2.objective / 1e6 # in Mio. €

28497.756665749355

## Exporting the model into a .nc file

In [ ]:
n_no_co2.export_to_netcdf("pypsa_model_n_no_co2.nc")